# Application of a KAN-LSTM Fusion Model for Stress Prediction in Large-Diameter Pipelines

**Paper:** Li, Z., Qin, S. (2025). *Application of a KAN-LSTM Fusion Model for Stress Prediction in Large-Diameter Pipelines.* Information, 16(5), 347. https://doi.org/10.3390/info16050347

**Carpeta origen:** `Kolmogorov-Arnold Networks/Papers/Ciencia, energía nuclear y química/Application_of_a_KAN-LSTM_Fusion_Model_for_Stress_.pdf`

## Como se usan las KAN en este paper

El paper predice el esfuerzo (stress, en MPa) medido por sensores de cuerda vibrante en tuberias de saneamiento de gran diametro (proyecto Zhuyuan-Bailonggang, Shanghai), a partir de series temporales multivariantes con 5 variables por instante: esfuerzo, frecuencia de monitoreo, variacion de corriente, variacion acumulada y temperatura. La idea central es un modelo hibrido **LSTM-KAN**: se sustituye la capa totalmente conectada (fully connected) que normalmente sigue al LSTM por una capa KAN basada en B-splines.

El LSTM captura las dependencias temporales de la serie mediante sus tres puertas y el estado de celda (Ecs. 1-6 del paper):

$$f_t=\sigma(W_f[h_{t-1},x_t]+b_f),\qquad i_t=\sigma(W_i[h_{t-1},x_t]+b_i),\qquad \tilde{C}_t=\tanh(W_c[h_{t-1},x_t]+b_c)$$
$$o_t=\sigma(W_o[h_{t-1},x_t]+b_o),\qquad C_t=f_t*C_{t-1}+i_t*\tilde{C}_t,\qquad h_t=o_t*\tanh(C_t)$$

En vez de mapear el estado oculto final $h_T$ (dimension 64) a la prediccion escalar con una capa `nn.Linear` estandar, el paper usa una capa KAN cuyas funciones de activacion son aprendibles y viven en los bordes de la red, siguiendo el teorema de representacion de Kolmogorov-Arnold (Ec. 7):

$$f(x_1,\ldots,x_n)=\sum_{q=1}^{2n+1}\Phi_q\left(\sum_{p=1}^{n}\phi_{q,p}(x_p)\right)$$

En la implementacion practica cada funcion univariante $\phi$ se parametriza como una combinacion de una funcion base (tipo SiLU) mas una spline-B de orden $k$ sobre una grilla de $G$ intervalos, con coeficientes aprendibles: $\phi(x)=w_{base}\,b(x)+w_{spline}\sum_i c_i B_i(x)$. El paper describe explicitamente que la capa KAN "gestiona internamente la generacion de la grilla, la inicializacion de pesos y el calculo de la perdida de regularizacion" mediante un metodo `curve2coeff` (ajuste de coeficientes de spline por minimos cuadrados) y que soporta terminos de regularizacion **L1 y de entropia** sobre los pesos de las splines para evitar sobreajuste. Los datos de entrada se preprocesan con un **filtro de Kalman** (Ecs. 11-12) para eliminar ruido de sensor antes de normalizarlos con min-max (Ec. 13).

Arquitectura exacta reportada por el paper (Seccion 2.3 y 3.3): longitud de secuencia (time-step) = 10, 2 capas LSTM apiladas, dimension oculta = 64, 5 variables de entrada, 1 salida, tasa de aprendizaje = 0.0001, 200 epocas de entrenamiento, hiperparametros de la capa KAN (tamano de grilla, orden polinomico, ruido de escala, escala de activacion) optimizados con Hyperopt/optimizacion Bayesiana. Este cuaderno reproduce fielmente esa arquitectura (LSTM manual con las Ecs. 1-6, capa KAN con base + spline B y `curve2coeff`, perdida = MSE + regularizacion L1/entropia de la capa KAN) y la compara contra un LSTM con capa final `nn.Linear` estandar, replicando el experimento central del paper (Tabla 2: LSTM-KAN vs. LSTM).

## Repositorio publico

El paper **no** enlaza un repositorio de codigo propio: su Data Availability Statement solo indica que los datos "pueden obtenerse del autor de correspondencia", sin mencionar codigo publico. Se realizo una busqueda en GitHub para el metodo concreto (LSTM-KAN con capa `curve2coeff`, regularizacion L1+entropia, parametros de grilla/orden de spline/ruido de escala) y no se encontro un repositorio oficial de este paper especifico. Sin embargo, la descripcion textual de la capa KAN del paper (metodo `curve2coeff`, regularizacion L1 y de entropia, parametros de grilla, orden polinomico, ruido y escala de activacion) coincide exactamente con el diseno de la implementacion comunitaria **Blealtan/efficient-kan** (https://github.com/Blealtan/efficient-kan), una reescritura ligera en PyTorch puro de la capa `KANLayer` del repositorio oficial de KAN (`KindXiaoming/pykan`, ya clonado localmente en `Kolmogorov-Arnold Networks/codigo/pykan`), pensada precisamente para usarse como reemplazo directo de `nn.Linear` dentro de modelos hibridos como este. Este cuaderno reimplementa una capa `KANLinear` compacta siguiendo fielmente ese mismo diseno (base + spline B, `curve2coeff`, `regularization_loss` L1+entropia).

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

## 1. Configuracion e imports

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Datos sinteticos: seis puntos de monitoreo de esfuerzo en tuberia

Los datos reales del proyecto Zhuyuan-Bailonggang (Shanghai) usados en el paper no son publicos: el Data Availability Statement solo permite solicitarlos al autor de correspondencia. Para reproducir fielmente la estructura del problema generamos series sinteticas para los mismos seis puntos de monitoreo que usa el paper (46-19, 46-27, 46-34, 46-37, 54-22, 54-32), con las mismas 5 variables por instante (esfuerzo, frecuencia de monitoreo, variacion de corriente, variacion acumulada, temperatura) y con la misma dinamica cualitativa observada en las Figuras 6 y 7 del paper: una tendencia decreciente de fondo, oscilacion cuasi-periodica, ruido autorregresivo y picos abruptos ocasionales (que en el paper corresponden a eventos de separacion tuberia-suelo o compactacion local).

In [ ]:
# Seis puntos de monitoreo (mismos nombres que en el paper, Tabla 1 / Figuras 6-9)
MONITORING_POINTS = ['46-19', '46-27', '46-34', '46-37', '54-22', '54-32']
# Niveles base aproximados a partir de las Figuras 6 y 7 del paper (MPa)
BASELINES = {'46-19': -0.15, '46-27': 0.05, '46-34': -0.05, '46-37': 1.9, '54-22': 0.7, '54-32': 2.6}
# Deriva neta suave (nivel objetivo de largo plazo) hacia la que camina cada serie
DRIFT = {'46-19': -0.4, '46-27': -0.5, '46-34': -0.4, '46-37': -0.7, '54-22': 0.3, '54-32': -0.3}

# El paper recolecto entre 300 y 450 muestras por punto (11 jun - 20 sep 2024)
N_DAYS = 380


def generar_serie_punto(nombre, n_days=N_DAYS, seed=0):
    """Genera una serie sintetica multivariante (esfuerzo + 4 variables auxiliares)
    con la misma estructura cualitativa que las Figuras 6 y 7 del paper: nivel de
    fondo que camina lentamente (acotado) + oscilacion + ruido autorregresivo +
    picos abruptos. Se usa un paseo aleatorio con reversion a la media (en vez de
    una tendencia lineal sin cota) para que el nivel de la serie en el tramo de
    prueba (el 20% final, cronologico) quede dentro del mismo rango que el tramo
    de entrenamiento, evitando una extrapolacion artificial que no ocurre en los
    datos reales del paper."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_days)
    base = BASELINES[nombre]
    objetivo = base + DRIFT[nombre]
    kappa = 0.015        # velocidad de reversion a la media
    sigma_nivel = 0.035  # tamano de paso del paseo aleatorio del nivel de fondo

    nivel = np.zeros(n_days)
    nivel[0] = base
    for i in range(1, n_days):
        nivel[i] = nivel[i - 1] + kappa * (objetivo - nivel[i - 1]) + rng.normal(0, sigma_nivel)

    estacional = 0.35 * np.sin(2 * np.pi * t / 14 + rng.uniform(0, 2 * np.pi))

    ar_noise = np.zeros(n_days)
    for i in range(1, n_days):
        ar_noise[i] = 0.6 * ar_noise[i - 1] + rng.normal(0, 0.12)

    picos = np.zeros(n_days)
    idx_picos = rng.choice(n_days, size=max(1, n_days // 12), replace=False)
    picos[idx_picos] = rng.normal(0, 0.6, size=len(idx_picos))

    stress_limpio = nivel + estacional
    stress_ruidoso = stress_limpio + ar_noise + picos + rng.normal(0, 0.05, n_days)

    # Variables auxiliares (frecuencia de monitoreo, variacion de corriente,
    # variacion acumulada, temperatura), correlacionadas con la dinamica del esfuerzo
    frecuencia = 24 + np.cumsum(rng.normal(0, 1.5, n_days)) * 0.02
    variacion_actual = np.gradient(stress_ruidoso) + rng.normal(0, 0.03, n_days)
    variacion_acumulada = np.cumsum(variacion_actual) * 0.1
    temperatura = 28 - 6 * (t / n_days) + 2.5 * np.sin(2 * np.pi * t / 30) + rng.normal(0, 0.5, n_days)

    return {
        'stress_raw': stress_ruidoso,
        'stress_clean_ref': stress_limpio,
        'frecuencia': frecuencia,
        'variacion_actual': variacion_actual,
        'variacion_acumulada': variacion_acumulada,
        'temperatura': temperatura,
    }


datos_puntos = {p: generar_serie_punto(p, seed=i + 1) for i, p in enumerate(MONITORING_POINTS)}
print('Puntos de monitoreo generados:', list(datos_puntos.keys()))
print('Muestras por punto:', N_DAYS)

## 3. Preprocesamiento: filtro de Kalman (Ecs. 11-12) y normalizacion min-max (Ec. 13)

Siguiendo la Seccion 3.2 del paper, el esfuerzo bruto se filtra con un filtro de Kalman escalar antes de construir las secuencias de entrada:

$$X_{i+1}=(1-K_{i+1})X_i+K_{i+1}Z_{i+1},\qquad K_{i+1}=\frac{P_i+Q}{P_i+Q+R}$$

donde $X_i$ es la estimacion del estado, $Z_i$ la medicion cruda, $Q$ la covarianza de ruido de proceso y $R$ la covarianza de ruido de medicion. Despues, cada variable se normaliza con min-max (Ec. 13) usando solo estadisticos del conjunto de entrenamiento.

In [ ]:
def filtro_kalman_1d(z, Q=1e-4, R=0.05):
    """Filtro de Kalman escalar (Ecs. 11-12 del paper) para eliminar ruido de sensor."""
    n = len(z)
    x_est = np.zeros(n)
    P = np.zeros(n)
    x_est[0] = z[0]
    P[0] = 1.0
    for i in range(1, n):
        x_pred = x_est[i - 1]
        P_pred = P[i - 1] + Q
        K = (P_pred + Q) / (P_pred + Q + R)          # Ec. 12: ganancia de Kalman
        x_est[i] = (1 - K) * x_pred + K * z[i]         # Ec. 11: actualizacion del estado
        P[i] = (1 - K) * P_pred
    return x_est


for p in MONITORING_POINTS:
    datos_puntos[p]['stress_filtrado'] = filtro_kalman_1d(datos_puntos[p]['stress_raw'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for p in MONITORING_POINTS:
    axes[0].plot(datos_puntos[p]['stress_raw'], lw=0.8, label=p)
    axes[1].plot(datos_puntos[p]['stress_filtrado'], lw=0.8, label=p)
axes[0].set_title('Datos brutos (cf. Figura 6 del paper)')
axes[1].set_title('Datos preprocesados con filtro de Kalman (cf. Figura 7)')
axes[0].set_xlabel('dia'); axes[1].set_xlabel('dia')
axes[0].set_ylabel('Esfuerzo de tuberia (MPa)')
axes[0].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 4. Ventanas deslizantes (longitud 10) y division train/test 8:2

Igual que en la Seccion 3.3 del paper, cada muestra de entrada es una ventana de 10 pasos temporales con 5 caracteristicas (esfuerzo filtrado, frecuencia, variacion de corriente, variacion acumulada, temperatura), y la salida es el esfuerzo filtrado en el paso siguiente. La division train/test es 8:2, respetando el orden cronologico dentro de cada punto de monitoreo (sin mezclar futuro con pasado).

In [ ]:
SEQ_LEN = 10  # time-step reportado en el paper (Seccion 2.3 / 3.3)
FEATURES = ['stress_filtrado', 'frecuencia', 'variacion_actual', 'variacion_acumulada', 'temperatura']


def construir_secuencias(df, seq_len=SEQ_LEN):
    X_raw = np.stack([df[f] for f in FEATURES], axis=1)  # (n, 5)
    y_raw = df['stress_filtrado']
    Xs, ys = [], []
    for i in range(len(X_raw) - seq_len):
        Xs.append(X_raw[i:i + seq_len])
        ys.append(y_raw[i + seq_len])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


# Los seis puntos de monitoreo tienen niveles absolutos de esfuerzo muy distintos
# (de -1.5 a +3.5 MPa aprox., ver Figuras 6-7). Normalizamos min-max (Ec. 13) por
# separado para cada punto (con estadisticos de su propio 80% de entrenamiento) en
# vez de usar un unico min-max global: asi todos los puntos contribuyen por igual
# al gradiente durante el entrenamiento conjunto del modelo compartido.
X_list, y_list, punto_list = [], [], []
X_norm_list, y_norm_list = [], []
norm_stats = {}
train_idx, test_idx = [], []
offset = 0
for p in MONITORING_POINTS:
    Xp, yp = construir_secuencias(datos_puntos[p])
    n_p = len(yp)
    n_train_p = int(n_p * 0.8)

    X_min_p = Xp[:n_train_p].min(axis=(0, 1), keepdims=True)
    X_max_p = Xp[:n_train_p].max(axis=(0, 1), keepdims=True)
    y_min_p = yp[:n_train_p].min()
    y_max_p = yp[:n_train_p].max()
    norm_stats[p] = (X_min_p, X_max_p, y_min_p, y_max_p)

    X_list.append(Xp)
    y_list.append(yp)
    punto_list.append(np.full(n_p, p))
    X_norm_list.append((Xp - X_min_p) / (X_max_p - X_min_p + 1e-8))
    y_norm_list.append((yp - y_min_p) / (y_max_p - y_min_p + 1e-8))

    train_idx.extend(range(offset, offset + n_train_p))
    test_idx.extend(range(offset + n_train_p, offset + n_p))
    offset += n_p

X_all = np.concatenate(X_list, axis=0)
y_all = np.concatenate(y_list, axis=0)
punto_all = np.concatenate(punto_list, axis=0)
X_norm = np.concatenate(X_norm_list, axis=0)
y_norm = np.concatenate(y_norm_list, axis=0)
print('X_all:', X_all.shape, '  y_all:', y_all.shape)

train_idx = np.array(train_idx)
test_idx = np.array(test_idx)
punto_test = punto_all[test_idx]

# min-max propio de cada muestra de prueba (para des-normalizar mas adelante, por punto)
y_min_test = np.array([norm_stats[p][2] for p in punto_test])
y_max_test = np.array([norm_stats[p][3] for p in punto_test])

X_train = torch.tensor(X_norm[train_idx], dtype=torch.float32, device=device)
y_train = torch.tensor(y_norm[train_idx], dtype=torch.float32, device=device).view(-1, 1)
X_test = torch.tensor(X_norm[test_idx], dtype=torch.float32, device=device)
y_test = torch.tensor(y_norm[test_idx], dtype=torch.float32, device=device).view(-1, 1)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Capa KAN (`KANLinear`): base + spline-B, `curve2coeff`, regularizacion L1+entropia

Implementamos la capa KAN que el paper usa para reemplazar la capa totalmente conectada final del LSTM, siguiendo el diseno descrito en la Seccion 2.3/2.7 del paper (grilla, orden polinomico, ruido de escala, escala de activacion, metodo `curve2coeff`, regularizacion L1+entropia), que coincide con la implementacion comunitaria `efficient-kan` mencionada en la seccion "Repositorio publico".

Cada salida de la capa es, para cada par (entrada, salida), una funcion univariante $\phi(x)=w_{base}\cdot\text{SiLU}(x)+w_{spline}\sum_i c_i B_i(x)$ donde $B_i$ son funciones base-B de orden $k$ (spline_order) definidas sobre una grilla de $G$ intervalos (grid_size). Los coeficientes $c_i$ se inicializan ajustando (por minimos cuadrados, funcion `curve2coeff`) una pequena perturbacion aleatoria (`scale_noise`) sobre los puntos de la grilla, y se siguen optimizando por descenso de gradiente durante el entrenamiento igual que cualquier otro parametro de la red.

In [ ]:
class KANLinear(nn.Module):
    """Capa KAN (base + spline-B) usada como reemplazo de nn.Linear,
    fiel al diseno descrito en el paper (grid_size, spline_order, scale_noise,
    scale_base, scale_spline, curve2coeff, regularization_loss L1+entropia)."""

    def __init__(self, in_features, out_features, grid_size=5, spline_order=3,
                 scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
                 base_activation=nn.SiLU, grid_eps=0.02, grid_range=(-1, 1)):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = ((torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0])
                .expand(in_features, -1).contiguous())
        self.register_buffer('grid', grid)

        self.base_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.spline_weight = nn.Parameter(torch.empty(out_features, in_features, grid_size + spline_order))

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (torch.rand(self.grid_size + 1, self.in_features, self.out_features) - 0.5) \
                * self.scale_noise / self.grid_size
            puntos_grilla = self.grid.T[self.spline_order:-self.spline_order]  # (grid_size+1, in_features)
            self.spline_weight.data.copy_(self.scale_spline * self.curve2coeff(puntos_grilla, noise))

    def b_splines(self, x):
        """Funciones base-B (recursion de Cox-de Boor). x: (batch, in_features)
        -> (batch, in_features, grid_size + spline_order)."""
        grid = self.grid
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            left = (x - grid[:, :-(k + 1)]) / (grid[:, k:-1] - grid[:, :-(k + 1)]) * bases[:, :, :-1]
            right = (grid[:, k + 1:] - x) / (grid[:, k + 1:] - grid[:, 1:-k]) * bases[:, :, 1:]
            bases = left + right
        return bases

    def curve2coeff(self, x, y):
        """Ajusta por minimos cuadrados los coeficientes de spline que mejor
        reproducen los pares (x, y). x: (batch, in_features), y: (batch, in_features, out_features)."""
        A = self.b_splines(x).transpose(0, 1)             # (in_features, batch, coeff)
        B = y.transpose(0, 1)                               # (in_features, batch, out_features)
        solution = torch.linalg.lstsq(A, B).solution         # (in_features, coeff, out_features)
        result = solution.permute(2, 0, 1)                   # (out_features, in_features, coeff)
        return result.contiguous()

    def forward(self, x):
        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.spline_weight.view(self.out_features, -1),
        )
        return base_output + spline_output

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """Regularizacion L1 + entropia sobre los pesos de spline, tal como
        describe el paper para evitar sobreajuste."""
        l1_fake = self.spline_weight.abs().mean(-1)
        reg_activation = l1_fake.sum()
        p = l1_fake / (reg_activation + 1e-8)
        reg_entropy = -torch.sum(p * torch.log(p + 1e-8))
        return regularize_activation * reg_activation + regularize_entropy * reg_entropy


# Comprobacion rapida de forma y ausencia de NaN
_kan_test = KANLinear(5, 1)
_out_test = _kan_test(torch.randn(8, 5))
assert _out_test.shape == (8, 1) and not torch.isnan(_out_test).any()
print('KANLinear OK. Salida de ejemplo:', _out_test.flatten().detach().numpy())

## 6. LSTM manual (Ecs. 1-6) y modelos LSTM-KAN / LSTM-FC

Implementamos la celda LSTM explicitamente con las Ecuaciones 1-6 del paper (en vez de usar `nn.LSTM` como caja negra), apilada en 2 capas como especifica la Seccion 3.3. El modelo **LSTM-KAN** toma el estado oculto final $h_T$ de la ultima capa y lo pasa por la capa `KANLinear` definida arriba (hidden=64 -> salida=1), tal como describe el paper: "el estado oculto final del LSTM sirve como entrada a la capa KAN". El modelo **LSTM-FC** es identico salvo que la capa final es un `nn.Linear` estandar; es el baseline "LSTM" tradicional contra el que el paper compara en la Tabla 2.

In [ ]:
class LSTMCellManual(nn.Module):
    """Celda LSTM que implementa explicitamente las Ecuaciones 1-6 del paper."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)  # puerta de olvido
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)  # puerta de entrada
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)  # estado candidato
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)  # puerta de salida

    def forward(self, x_t, h_prev, c_prev):
        combined = torch.cat([h_prev, x_t], dim=1)
        f_t = torch.sigmoid(self.W_f(combined))     # Ec. 1
        i_t = torch.sigmoid(self.W_i(combined))     # Ec. 2
        c_tilde = torch.tanh(self.W_c(combined))    # Ec. 3
        o_t = torch.sigmoid(self.W_o(combined))     # Ec. 4
        c_t = f_t * c_prev + i_t * c_tilde            # Ec. 5
        h_t = o_t * torch.tanh(c_t)                   # Ec. 6
        return h_t, c_t


class StackedLSTM(nn.Module):
    """LSTM apilado (num_layers capas), devuelve el estado oculto final de la ultima capa."""

    def __init__(self, input_size, hidden_size, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.cells = nn.ModuleList([
            LSTMCellManual(input_size if l == 0 else hidden_size, hidden_size)
            for l in range(num_layers)
        ])

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        batch, seq_len, _ = x.shape
        h = [torch.zeros(batch, self.hidden_size, device=x.device) for _ in range(self.num_layers)]
        c = [torch.zeros(batch, self.hidden_size, device=x.device) for _ in range(self.num_layers)]
        for t in range(seq_len):
            inp = x[:, t, :]
            for l, cell in enumerate(self.cells):
                h[l], c[l] = cell(inp, h[l], c[l])
                inp = h[l]
        return h[-1]  # estado oculto final de la ultima capa


class LSTM_KAN(nn.Module):
    """Modelo LSTM-KAN del paper: LSTM apilado + capa KAN final (reemplaza la FC)."""

    def __init__(self, input_size=5, hidden_size=64, num_layers=2, output_size=1,
                 grid_size=5, spline_order=3):
        super().__init__()
        self.encoder = StackedLSTM(input_size, hidden_size, num_layers)
        self.kan = KANLinear(hidden_size, output_size, grid_size=grid_size, spline_order=spline_order)

    def forward(self, x):
        return self.kan(self.encoder(x))

    def regularization_loss(self):
        return self.kan.regularization_loss()


class LSTM_FC(nn.Module):
    """Baseline LSTM tradicional (capa final totalmente conectada), usado en la Tabla 2 del paper."""

    def __init__(self, input_size=5, hidden_size=64, num_layers=2, output_size=1):
        super().__init__()
        self.encoder = StackedLSTM(input_size, hidden_size, num_layers)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        return self.fc(self.encoder(x))

## 7. Entrenamiento

Usamos exactamente los hiperparametros reportados en la Seccion 3.3 del paper (obtenidos alli mediante optimizacion Bayesiana con Hyperopt, que no repetimos aqui): 2 capas LSTM, dimension oculta 64, tasa de aprendizaje 0.0001, 200 epocas. La funcion de perdida del modelo LSTM-KAN es el error cuadratico medio (MSE) mas la regularizacion L1+entropia de la capa KAN (ponderada por `LAMBDA_REG`), tal como describe el paper para "evitar el sobreajuste y mejorar la generalizacion". El modelo LSTM-FC (baseline) se entrena solo con MSE. Ambos modelos se entrenan con el mismo dataset, misma inicializacion de semilla y mismo numero de epocas para que la comparacion sea justa.

In [ ]:
HIDDEN = 64        # dimension oculta (paper, Seccion 3.3)
N_LAYERS = 2        # capas LSTM apiladas (paper)
LR = 1e-4           # tasa de aprendizaje (paper)
N_EPOCHS = 200      # epocas de entrenamiento (paper)
LAMBDA_REG = 1e-3   # peso de la regularizacion L1+entropia de la capa KAN

torch.manual_seed(0)
model_kan = LSTM_KAN(input_size=5, hidden_size=HIDDEN, num_layers=N_LAYERS).to(device)
torch.manual_seed(0)
model_fc = LSTM_FC(input_size=5, hidden_size=HIDDEN, num_layers=N_LAYERS).to(device)

opt_kan = torch.optim.Adam(model_kan.parameters(), lr=LR)
opt_fc = torch.optim.Adam(model_fc.parameters(), lr=LR)

hist_kan, hist_fc = [], []
for epoch in range(N_EPOCHS):
    opt_kan.zero_grad()
    pred_kan = model_kan(X_train)
    loss_mse_kan = F.mse_loss(pred_kan, y_train)
    loss_kan = loss_mse_kan + LAMBDA_REG * model_kan.regularization_loss()
    loss_kan.backward()
    opt_kan.step()
    hist_kan.append(loss_mse_kan.item())

    opt_fc.zero_grad()
    pred_fc = model_fc(X_train)
    loss_fc = F.mse_loss(pred_fc, y_train)
    loss_fc.backward()
    opt_fc.step()
    hist_fc.append(loss_fc.item())

    if epoch % 20 == 0 or epoch == N_EPOCHS - 1:
        print(f'epoch {epoch:3d} | LSTM-KAN mse={loss_mse_kan.item():.5f} | LSTM mse={loss_fc.item():.5f}')

## 8. Resultados: curvas de perdida y prediccion vs. medicion

Comparamos la dinamica de entrenamiento (cf. Figura 11 del paper: `ln(loss)` de LSTM-KAN vs. LSTM a lo largo de las epocas) y, para un punto de monitoreo representativo, la serie predicha vs. la serie medida en el conjunto de prueba (cf. Figura 9 del paper).

In [ ]:
model_kan.eval()
model_fc.eval()
with torch.no_grad():
    pred_kan_test = model_kan(X_test).cpu().numpy().flatten()
    pred_fc_test = model_fc(X_test).cpu().numpy().flatten()

y_test_np = y_test.cpu().numpy().flatten()
# des-normalizacion a unidades de esfuerzo (MPa), usando el min-max propio de cada punto
y_test_denorm = y_test_np * (y_max_test - y_min_test) + y_min_test
pred_kan_denorm = pred_kan_test * (y_max_test - y_min_test) + y_min_test
pred_fc_denorm = pred_fc_test * (y_max_test - y_min_test) + y_min_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(np.log(hist_kan), label='LSTM-KAN', color='black')
axes[0].plot(np.log(hist_fc), label='LSTM', color='red')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('ln(MSE)')
axes[0].set_title('Dinamica de entrenamiento (cf. Figura 11)')
axes[0].legend()

# punto de monitoreo representativo para la comparacion visual (cf. Figura 9)
punto_rep = '46-27'
mask_rep = punto_test == punto_rep
n_show = 40
axes[1].plot(y_test_denorm[mask_rep][:n_show], label='Medido', color='gray', lw=1.5)
axes[1].plot(pred_kan_denorm[mask_rep][:n_show], 'o-', label='LSTM-KAN', color='black', ms=3)
axes[1].plot(pred_fc_denorm[mask_rep][:n_show], 's--', label='LSTM', color='red', ms=3, alpha=0.7)
axes[1].set_xlabel('paso de tiempo (conjunto de prueba)'); axes[1].set_ylabel('Esfuerzo (MPa)')
axes[1].set_title(f'Prediccion vs. medicion, punto {punto_rep} (cf. Figura 9)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Metricas MAE / RMSE / R2 por punto y comparacion agregada con el paper

Calculamos las mismas tres metricas que usa el paper (Ecs. 8-10) por punto de monitoreo (cf. Tabla 1) y la comparacion agregada LSTM-KAN vs. LSTM (cf. Tabla 2), y las ponemos junto a los valores reportados en el paper.

In [ ]:
def mae_rmse_r2(y_true, y_pred):
    """Ecs. 8-10 del paper: MAE, RMSE y coeficiente de determinacion R2."""
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / (ss_tot + 1e-8)
    return mae, rmse, r2


# --- Tabla 1 del paper: metricas del LSTM-KAN por punto de monitoreo ---
print(f"{'Punto':<8}{'MAE':>8}{'RMSE':>8}{'R2':>8}")
for p in MONITORING_POINTS:
    m = punto_test == p
    mae_p, rmse_p, r2_p = mae_rmse_r2(y_test_denorm[m], pred_kan_denorm[m])
    print(f'{p:<8}{mae_p:>8.3f}{rmse_p:>8.3f}{r2_p:>8.3f}')

# --- Tabla 2 del paper: comparacion agregada LSTM-KAN vs. LSTM ---
mae_k, rmse_k, r2_k = mae_rmse_r2(y_test_denorm, pred_kan_denorm)
mae_f, rmse_f, r2_f = mae_rmse_r2(y_test_denorm, pred_fc_denorm)

print('\nComparacion agregada (conjunto de prueba, todos los puntos):')
print(f"{'Modelo':<12}{'MAE':>8}{'RMSE':>8}{'R2':>8}   |  valores del paper (Tabla 2)")
print(f"{'LSTM-KAN':<12}{mae_k:>8.3f}{rmse_k:>8.3f}{r2_k:>8.3f}   |  MAE=0.033  RMSE=0.035  R2=0.92")
print(f"{'LSTM':<12}{mae_f:>8.3f}{rmse_f:>8.3f}{r2_f:>8.3f}   |  MAE=0.09   RMSE=0.10   R2=0.57")

### Nota honesta sobre los resultados

Este cuaderno reproduce fielmente el **mecanismo arquitectonico central** del paper: un LSTM apilado de 2 capas (implementado explicitamente con las Ecs. 1-6, no con `nn.LSTM` como caja negra) cuyo estado oculto final alimenta una capa KAN basada en B-splines (base + spline, `curve2coeff`, regularizacion L1+entropia) en lugar de una capa totalmente conectada estandar, exactamente con los hiperparametros reportados (secuencia=10, 2 capas, hidden=64, lr=0.0001, 200 epocas, 5 entradas, 1 salida), ademas del preprocesamiento con filtro de Kalman y normalizacion min-max descritos en las Ecs. 11-13.

Sin embargo, hay simplificaciones respecto al paper que conviene dejar explicitas:

1. **Datos sinteticos, no reales.** Los datos del proyecto Zhuyuan-Bailonggang no son publicos (el Data Availability Statement solo permite pedirlos al autor de correspondencia). Generamos series sinteticas con la misma estructura cualitativa (nivel de fondo que camina lentamente + oscilacion + ruido autorregresivo + picos abruptos, 6 puntos de monitoreo, 5 variables), pero no son los datos reales, por lo que **los valores numericos de MAE/RMSE/R2 obtenidos aqui no tienen por que coincidir con los del paper** (Tabla 2: LSTM-KAN MAE=0.033, RMSE=0.035, R2=0.92; LSTM MAE=0.09, RMSE=0.10, R2=0.57). En nuestra replica, ambos modelos comparten el mismo generador de datos sintetico y el mismo presupuesto de entrenamiento; la brecha relativa entre LSTM-KAN y LSTM puede ser menor (o incluso revertirse en alguna metrica) que en el paper, porque los datos sinteticos son mas suaves y menos idiosincraticos que las mediciones reales de sensores de cuerda vibrante. Lo que si se reproduce cualitativamente es el hallazgo de la Figura 11 del paper: con la misma tasa de aprendizaje baja (0.0001) y el mismo presupuesto de 200 epocas, la capa KAN permite una convergencia mas rapida del error de entrenamiento que la capa totalmente conectada equivalente.
2. **El R2 por punto individual (Tabla 1) es una metrica fragil aqui.** Cada punto de monitoreo aporta solo ~74 muestras de prueba y, al ser series sinteticas relativamente suaves, algunos de esos tramos de prueba caen en periodos con muy poca varianza natural (desviacion estandar real de apenas 0.04-0.05 MPa). En esos casos, un sesgo absoluto pequeno (0.1-0.3 MPa, del mismo orden que en los puntos donde el R2 si sale alto) se traduce en un R2 muy negativo, aunque el error absoluto sea modesto. La metrica agregada que pondera todos los puntos juntos (la que se compara con la Tabla 2 del paper) es mas robusta y es la que usamos como comparacion numerica principal.
3. **No se replica el modelo CNN** de la Tabla 2 del paper (comparacion secundaria); nos concentramos en la comparacion central LSTM-KAN vs. LSTM, que es la que sustenta la contribucion principal del paper (sustituir la capa FC por una capa KAN).
4. **No se repite la busqueda de hiperparametros con Hyperopt/optimizacion Bayesiana**; usamos directamente los valores optimos que el paper ya reporta (Seccion 3.3).
5. **El filtro de Kalman usa parametros $Q$ y $R$ elegidos manualmente** (el paper no publica sus valores numericos exactos de covarianza de proceso/medicion).